In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pandas as pd
import pickle


In [ ]:
df=pd.read_csv("Churn_Modelling.csv")
df.head()

In [ ]:
##preprocessing the data
## droping the irrevelent columns
df=df.drop(['CustomerId','RowNumber','Surname'],axis=1)
df.head()

In [ ]:
##encoding the categorical variable
label_encoder_gender=LabelEncoder()
df['Gender']=label_encoder_gender.fit_transform(df['Gender'])
df['Gender']


In [ ]:
## onehot encode geographical column
from sklearn.preprocessing import OneHotEncoder
geo_ohe=OneHotEncoder()
geo_encoder=geo_ohe.fit_transform(df[['Geography']])
geo_encoder


In [ ]:
geo_ohe.get_feature_names_out(['Geography'])

In [ ]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=geo_ohe.get_feature_names_out(['Geography']))
geo_encoded_df

In [ ]:
df=pd.concat([df.drop('Geography',axis=1),geo_encoded_df],axis=1)

In [ ]:
df

In [ ]:
X=df.drop('Exited',axis=1);
y=df['Exited']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
##scale the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)



In [ ]:
X_train

In [ ]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

In [ ]:
with open('onehot_encoder_geography.pkl','wb') as file:
    pickle.dump(geo_ohe,file)

In [ ]:
with open("scaler.pkl",'wb') as file:
    pickle.dump(scaler,file)


ANN implementation

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

In [ ]:
##ann model building

model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
]
)

In [ ]:
model.summary()

In [ ]:
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()



In [ ]:
##compile the model

model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

In [ ]:
import datetime

In [ ]:
log_dir='logs/fit/'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [ ]:
##set up early stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [ ]:
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),
    epochs=100,callbacks=[tensorflow_callback,early_stopping_callback]
)

In [ ]:
model.save('model.h5')

In [2]:
##load tensorflow extension
%load_ext tensorboard

In [3]:
%tensorboard  --logdir logs/fit  --port 6007

Reusing TensorBoard on port 6007 (pid 35732), started 11:51:57 ago. (Use '!kill 35732' to kill it.)